# 07 — Deep Learning Anomaly Detection

## 1. Objective

This notebook develops and examines deep-learning models for unsupervised anomaly detection in the Secure Water Treatment (SWaT) dataset.

The analysis builds on the data preparation, chronological partitioning, and LSTM Autoencoder work completed in the preceding notebooks. It focuses on two additional architectures: a Temporal Convolutional Network Autoencoder (TCN Autoencoder) and Unsupervised Anomaly Detection on Multivariate Time Series (USAD).

The objective is to investigate whether these architectures can learn patterns from normal industrial process observations and assign higher anomaly scores to observations that differ from those learned patterns.

## 2. Scope

This notebook covers:

- TCN Autoencoder methodology, implementation, training, and reconstruction-score analysis.
- USAD methodology, implementation, training, and anomaly-score analysis.
- Documentation of model configurations, training behaviour, computational requirements, and observed limitations.
- Preservation and verification of the resulting model artifacts and relevant outputs.

The LSTM Autoencoder developed in Notebook 05 remains part of the project's deep-learning model set. Its existing implementation and artifacts will be reused where appropriate rather than unnecessarily reproduced.

## 3. Methodological Principles

All implementations must follow the project's established data and evaluation protocol.

- Train the models using the designated normal training partition.
- Preserve chronological separation between training, threshold-calibration, and validation observations.
- Reuse the established 42-feature definition and training-fitted scaler where appropriate.
- Do not use attack labels to fit models, select model configurations, or determine anomaly thresholds.
- Record actual training outputs, score distributions, and computational costs without inventing results.
- Document observations and limitations before proceeding to the next stage.
- Keep final threshold selection and formal evaluation in Notebook 08.

## 4. Expected Outcome

The notebook will provide reproducible implementations and documented results for the TCN Autoencoder and USAD. Together with the existing LSTM Autoencoder, these models will form the project's deep-learning detector set for subsequent threshold calibration, comparative evaluation, and error analysis.

## 5. TCN Autoencoder — Methodological Foundations

### 5.1 Objective

The Temporal Convolutional Network Autoencoder (TCN Autoencoder) is an unsupervised deep-learning model that learns to reconstruct multivariate time-series windows from normal industrial process observations.

Unlike the LSTM Autoencoder, which uses recurrent layers to represent temporal dependencies, the TCN Autoencoder uses temporal convolutions. Dilated convolutions allow the network to incorporate information from a wider temporal context without requiring a recurrent operation at every timestep.

The model will be trained using the designated normal training partition. Its reconstruction errors will provide anomaly-oriented scores for subsequent analysis.

### 5.2 Input Representation

Each model input is a multivariate time-series window:

$$
X \in \mathbb{R}^{W \times F}
$$

where:

- $W = 30$ is the established sequence-window length.
- $F = 42$ is the number of selected sensor and actuator features.

A batch of windows is represented as:

$$
X \in \mathbb{R}^{B \times W \times F}
$$

where $B$ is the batch size.

The implementation must account for PyTorch's expected input layout for one-dimensional convolutions, which is typically:

$$
X \in \mathbb{R}^{B \times F \times W}
$$

Any necessary dimension permutation must be explicit and verified.

### 5.3 Temporal Convolutions and Dilation

A temporal convolution applies learnable filters across the time dimension. Dilation introduces gaps between filter positions, allowing a convolutional layer to access a wider temporal range without proportionally increasing the kernel size.

For a one-dimensional dilated causal convolution, the operation is:

$$
y_t = \sum_{i=0}^{K-1} w_i \, x_{t - d \cdot i}
$$

where:

- $x_t$ is the input sequence.
- $y_t$ is the output at timestep $t$.
- $w_i$ represents a learnable filter coefficient.
- $K$ is the kernel size.
- $d$ is the dilation factor.

The convolution is causal when the input is padded with $(K-1) \cdot d$ zeros on its left edge and the padded sequence is indexed from $1$; equivalently, $x_p = 0$ for all $p \leq 0$. Under that padding, the output $y_t$ depends only on $x_t$ and the preceding positions $x_{t-d}, x_{t-2d}, \ldots, x_{t-(K-1)d}$. No future observation $x_{t+j}$ with $j > 0$ contributes to $y_t$.

### 5.4 Receptive Field

The receptive field is the number of input timesteps that can influence an output position.

For a stack of $L$ dilated convolutional layers with common kernel size $K$ and dilations $d_1, d_2, \ldots, d_L$, the receptive field is:

$$
R = 1 + (K-1) \sum_{\ell=1}^{L} d_\ell
$$

$L$ in this formula is the total number of convolutional layers, not the number of TCN blocks. If a block contains two convolutions with the same dilation factor, its contribution to the sum is $2 \cdot d$ and to $L$ is $2$. This distinction matters when comparing against published configurations, which typically stack 2 convolutions per block.

For the reference configuration with four TCN blocks, dilation schedule $1, 2, 4, 8$, and kernel size $3$:

- If each block contains 1 convolution ($L = 4$, $d = [1, 2, 4, 8]$): $R = 1 + 2 \cdot (1+2+4+8) = 31$.
- If each block contains 2 convolutions ($L = 8$, $d = [1,1,2,2,4,4,8,8]$): $R = 1 + 2 \cdot 30 = 61$.
- If each block contains 3 convolutions ($L = 12$, $d = [1,1,1,2,2,2,4,4,4,8,8,8]$): $R = 1 + 2 \cdot 45 = 91$.

The reference reports a receptive field of 90 timesteps. That value is consistent with 3 convolutions per block (giving 91) but not with 1 or 2; it must be verified against the actual implementation, not inferred from the dilation schedule alone. The count reported here assumes no stride, no downsampling between blocks, and no extra conv layers on the residual path — any of those would change the effective receptive field.

Because this project's established input window contains 30 timesteps, the implementation must also be checked to confirm how its receptive field compares with the window length. If the computed receptive field exceeds 30, the network's output at early window positions will be computed from padded input only, and its reconstruction of those positions may be less informative than the middle and end of the window.

### 5.5 Encoder and Decoder

The encoder transforms the input sequence into a more compact representation of its temporal patterns. Dilated convolutional layers allow the encoder to represent dependencies across multiple temporal scales.

The decoder maps the encoded representation back to the original sequence dimensions.

For an input window $X$, the reconstructed output is denoted by:

$$
\hat{X} = f_{\theta}(X)
$$

where $f_{\theta}$ represents the trainable encoder–decoder network.

The reconstructed output must preserve the input window's temporal and feature dimensions:

$$
\hat{X} \in \mathbb{R}^{B \times W \times F}
$$

This shape requirement will be verified before training.

### 5.6 Reconstruction Loss

The model uses mean squared error (MSE) as its reconstruction objective:

$$
\mathcal{L}_{\mathrm{MSE}}
=
\frac{1}{B W F}
\sum_{b=1}^{B}
\sum_{t=1}^{W}
\sum_{f=1}^{F}
\left( X_{b,t,f} - \hat{X}_{b,t,f} \right)^2
$$

where:

- $X_{b,t,f}$ is the observed feature value.
- $\hat{X}_{b,t,f}$ is its reconstructed value.
- $B$, $W$, and $F$ denote batch size, window length, and feature count.

Minimizing this loss encourages the model to reconstruct the patterns represented in its training data.

### 5.7 Anomaly Scoring

After training, reconstruction error is calculated for each evaluated sequence.

For sequence $i$, the mean squared reconstruction error is:

$$
s_i =
\frac{1}{W F}
\sum_{t=1}^{W}
\sum_{f=1}^{F}
\left( X_{i,t,f} - \hat{X}_{i,t,f} \right)^2
$$

Higher scores indicate greater reconstruction error and are interpreted as greater anomalousness under this scoring convention.

High reconstruction error does not by itself establish a physical fault or cyberattack. Formal classifications will depend on the threshold-calibration and evaluation protocol defined in Notebook 08.

### 5.8 Training and Evaluation Controls

The implementation preserves the project's existing methodological controls:

- Use the established normal training partition for model fitting.
- Reuse the existing 42-feature definition and training-fitted scaler.
- Preserve chronological separation between training, threshold-calibration, and validation data.
- Avoid using attack labels for model fitting, architecture selection, or threshold selection.
- Verify model input and output dimensions before training.
- Record the actual configuration, training history, runtime, and reconstruction-score distributions.
- Save the fitted model and verify that it can be reloaded.
- Defer final threshold selection and formal evaluation to Notebook 08.

The TCN Autoencoder will be assessed using recorded evidence from this implementation. Results from the reference notebook will not be assumed to transfer to the current dataset or configuration.

In [1]:
# 6. TCN AUTOENCODER — REQUIRED ARTIFACT VERIFICATION

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

ARTIFACT_DIR = Path("/content/drive/MyDrive/swat-artifacts")
SCALER_PATH = ARTIFACT_DIR / "scaler.joblib"

if not ARTIFACT_DIR.is_dir():
    raise FileNotFoundError(
        f"Artifact directory not found: {ARTIFACT_DIR}"
    )

if not SCALER_PATH.is_file():
    raise FileNotFoundError(
        f"Training-fitted scaler not found: {SCALER_PATH}"
    )

print("Artifact verification: PASS")
print(f"Artifact directory: {ARTIFACT_DIR}")
print(f"Scaler file: {SCALER_PATH}")
print(f"Scaler size: {SCALER_PATH.stat().st_size / (1024 ** 2):.2f} MB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Artifact verification: PASS
Artifact directory: /content/drive/MyDrive/swat-artifacts
Scaler file: /content/drive/MyDrive/swat-artifacts/scaler.joblib
Scaler size: 0.00 MB


In [3]:
# 6.1 LOAD AND VERIFY THE EXISTING SCALER

import joblib
import numpy as np

# Load the scaler fitted on the designated training partition.
scaler = joblib.load(SCALER_PATH)

# Recover the established feature names and their ordering.
if not hasattr(scaler, "feature_names_in_"):
    raise AttributeError(
        "The saved scaler does not contain feature_names_in_. "
        "The established feature ordering must be verified separately."
    )

FEATURE_COLUMNS = list(scaler.feature_names_in_)

# Verify the expected feature count.
assert len(FEATURE_COLUMNS) == 42, (
    f"Expected 42 features, found {len(FEATURE_COLUMNS)}."
)

# Verify feature names are unique.
assert len(FEATURE_COLUMNS) == len(set(FEATURE_COLUMNS)), (
    "Duplicate feature names detected."
)

# Verify the scaler's fitted dimensionality.
assert scaler.n_features_in_ == len(FEATURE_COLUMNS), (
    "Scaler feature count does not match its recorded feature names."
)

print("Scaler verification: PASS")
print(f"Scaler type: {type(scaler).__name__}")
print(f"Number of features: {len(FEATURE_COLUMNS)}")
print(f"Fitted feature count: {scaler.n_features_in_}")
print(f"Feature names and order: {FEATURE_COLUMNS}")

Scaler verification: PASS
Scaler type: MinMaxScaler
Number of features: 42
Fitted feature count: 42
Feature names and order: ['FIT101', 'LIT101', 'MV101', 'P101', 'P102', 'AIT201', 'AIT202', 'AIT203', 'FIT201', 'MV201', 'P201', 'P203', 'P205', 'DPIT301', 'FIT301', 'LIT301', 'MV301', 'MV302', 'MV303', 'MV304', 'P302', 'AIT401', 'AIT402', 'FIT401', 'LIT401', 'P402', 'P403', 'UV401', 'AIT501', 'AIT502', 'AIT503', 'AIT504', 'FIT501', 'FIT502', 'FIT503', 'FIT504', 'P501', 'PIT501', 'PIT502', 'PIT503', 'FIT601', 'P602']


In [4]:
from pathlib import Path
import json

METADATA_PATH = ARTIFACT_DIR / "traditional_ml_model_metadata.json"

print(f"Metadata file exists: {METADATA_PATH.is_file()}")

if METADATA_PATH.is_file():
    with open(METADATA_PATH, "r", encoding="utf-8") as file:
        metadata = json.load(file)

    print("Metadata type:", type(metadata).__name__)

    if isinstance(metadata, dict):
        print("Available metadata keys:")
        for key in metadata:
            print(f"- {key}")
else:
    print("Traditional ML metadata is unavailable; it is not required for TCN training.")

Metadata file exists: True
Metadata type: dict
Available metadata keys:
- notebook
- scikit_learn_version
- feature_count
- feature_columns
- training_observations
- training_features
- models


In [5]:
for path in sorted(ARTIFACT_DIR.iterdir()):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"{path.name:<45} {size_mb:>10.2f} MB")
    elif path.is_dir():
        print(f"{path.name}/")

evaluation_sequences.npz                           10.24 MB
isolation_forest.joblib                             2.27 MB
lof_model.joblib                                   54.05 MB
lstm_autoencoder.pth                                0.51 MB
scaler.joblib                                       0.00 MB
sgd_one_class_svm.joblib                            0.00 MB
traditional_ml_model_metadata.json                  0.00 MB


In [6]:
from pathlib import Path

NORMAL_PATH = Path("/content/drive/MyDrive/swat-dataset/normal.csv")

if NORMAL_PATH.is_file():
    print("Normal dataset: FOUND")
    print(f"Path: {NORMAL_PATH}")
    print(f"Size: {NORMAL_PATH.stat().st_size / (1024 ** 2):.2f} MB")
else:
    print("Normal dataset: NOT FOUND")
    print(f"Expected path: {NORMAL_PATH}")

Normal dataset: FOUND
Path: /content/drive/MyDrive/swat-dataset/normal.csv
Size: 383.73 MB


In [7]:
import numpy as np

SEQUENCE_PATH = ARTIFACT_DIR / "evaluation_sequences.npz"

with np.load(SEQUENCE_PATH, allow_pickle=False) as sequences:
    print("Saved sequence arrays:")

    for name in sequences.files:
        array = sequences[name]
        print(
            f"{name}: "
            f"shape={array.shape}, "
            f"dtype={array.dtype}, "
            f"finite={np.isfinite(array).all()}"
        )

Saved sequence arrays:
X_threshold_seq: shape=(47408, 30, 42), dtype=float32, finite=True
X_validation_seq: shape=(79031, 30, 42), dtype=float32, finite=True
threshold_end_indices: shape=(47408,), dtype=int64, finite=True
validation_end_indices: shape=(79031,), dtype=int64, finite=True


In [9]:
with np.load(SEQUENCE_PATH, allow_pickle=False) as saved:
    threshold_indices = saved["threshold_end_indices"].copy()
    validation_indices = saved["validation_end_indices"].copy()

for name, indices in (
    ("threshold_end_indices", threshold_indices),
    ("validation_end_indices", validation_indices),
):
    print(f"\n{name}")
    print(f"Count: {len(indices):,}")
    print(f"First index: {indices[0]:,}")
    print(f"Last index: {indices[-1]:,}")
    print(f"Strictly increasing: {np.all(np.diff(indices) > 0)}")


threshold_end_indices
Count: 47,408
First index: 29
Last index: 47,436
Strictly increasing: True

validation_end_indices
Count: 79,031
First index: 29
Last index: 79,059
Strictly increasing: True


In [10]:
for name, indices in (
    ("Threshold", threshold_indices),
    ("Validation", validation_indices),
):
    differences, counts = np.unique(np.diff(indices), return_counts=True)

    print(f"\n{name} sequence indices")
    print(f"First index: {indices[0]:,}")
    print(f"Last index: {indices[-1]:,}")
    print(f"Minimum difference: {np.diff(indices).min():,}")
    print(f"Maximum difference: {np.diff(indices).max():,}")
    print("Index-step frequencies:")

    for difference, count in zip(differences, counts):
        print(f"  Step {difference}: {count:,} occurrences")


Threshold sequence indices
First index: 29
Last index: 47,436
Minimum difference: 1
Maximum difference: 1
Index-step frequencies:
  Step 1: 47,407 occurrences

Validation sequence indices
First index: 29
Last index: 79,059
Minimum difference: 1
Maximum difference: 1
Index-step frequencies:
  Step 1: 79,030 occurrences


In [11]:
print("Notebook:", metadata.get("notebook"))
print("Recorded feature count:", metadata.get("feature_count"))
print("Recorded training observations:", metadata.get("training_observations"))

recorded_features = metadata.get("feature_columns", [])

print("Recorded feature list length:", len(recorded_features))
print("Matches scaler feature order:", recorded_features == FEATURE_COLUMNS)

Notebook: 06_traditional_ml_anomaly_detection
Recorded feature count: 42
Recorded training observations: 221366
Recorded feature list length: 42
Matches scaler feature order: True


In [13]:
import pandas as pd
import numpy as np
import warnings

TRAIN_ROWS = 221_366
MONITOR_ROWS = 47_435
THRESHOLD_ROWS = 47_437
VALIDATION_ROWS = 79_060

ROWS_NEEDED = TRAIN_ROWS + MONITOR_ROWS
TIMESTAMP_COL = "Timestamp"
CHUNK_SIZE = 100_000

warnings.filterwarnings("ignore", message=".*Parsing dates.*")

# Read the header only to identify the sensor columns.
normal_header = pd.read_csv(NORMAL_PATH, nrows=0).columns.str.strip().tolist()

if TIMESTAMP_COL not in normal_header:
    raise KeyError("The normal dataset does not contain 'Timestamp'.")

LABEL_COL = next(
    (
        column
        for column in normal_header
        if column.lower() in {"normal/attack", "label"}
    ),
    None,
)

if LABEL_COL is None:
    raise KeyError("The normal dataset's label column was not found.")

SENSOR_COLUMNS = [
    column
    for column in normal_header
    if column not in {TIMESTAMP_COL, LABEL_COL}
]

if not set(FEATURE_COLUMNS).issubset(SENSOR_COLUMNS):
    raise ValueError("The scaler's 42 features do not match the CSV schema.")

eligible_parts = []
eligible_rows_collected = 0

for chunk in pd.read_csv(
    NORMAL_PATH,
    chunksize=CHUNK_SIZE,
    low_memory=False,
):
    chunk.columns = chunk.columns.str.strip()

    # Parse timestamps and retain only valid, complete observations.
    chunk[TIMESTAMP_COL] = pd.to_datetime(
        chunk[TIMESTAMP_COL],
        errors="coerce",
    )

    valid_mask = (
        chunk[TIMESTAMP_COL].notna()
        & chunk[SENSOR_COLUMNS].notna().all(axis=1)
    )

    eligible_chunk = chunk.loc[valid_mask].copy()

    if eligible_chunk.empty:
        continue

    # Retain only as many eligible rows as are still required.
    rows_remaining = ROWS_NEEDED - eligible_rows_collected
    eligible_chunk = eligible_chunk.iloc[:rows_remaining]

    eligible_parts.append(
        eligible_chunk[[TIMESTAMP_COL] + SENSOR_COLUMNS]
    )

    eligible_rows_collected += len(eligible_chunk)

    if eligible_rows_collected >= ROWS_NEEDED:
        break

if eligible_rows_collected != ROWS_NEEDED:
    raise ValueError(
        f"Expected {ROWS_NEEDED:,} eligible observations, "
        f"but recovered {eligible_rows_collected:,}."
    )

# Combine only the rows needed for training and monitoring.
training_monitor_df = pd.concat(
    eligible_parts,
    ignore_index=True,
)

# Confirm that the reconstructed rows preserve chronological order.
if not training_monitor_df[TIMESTAMP_COL].is_monotonic_increasing:
    raise ValueError(
        "The reconstructed timestamps are not monotonically increasing. "
        "Stop and investigate before training."
    )

# Verify completeness across all sensor columns.
if training_monitor_df[SENSOR_COLUMNS].isna().any().any():
    raise ValueError("Missing sensor values remain in the reconstructed data.")

# Create the established chronological partitions.
training_df = training_monitor_df.iloc[:TRAIN_ROWS].copy()
monitor_df = training_monitor_df.iloc[TRAIN_ROWS:].copy()

assert len(training_df) == TRAIN_ROWS
assert len(monitor_df) == MONITOR_ROWS

print("Training and monitoring data reconstruction: PASS")
print(f"Eligible observations recovered: {len(training_monitor_df):,}")
print(f"Training observations: {len(training_df):,}")
print(f"Monitoring observations: {len(monitor_df):,}")
print(f"All sensor columns: {len(SENSOR_COLUMNS)}")
print(f"Selected model features: {len(FEATURE_COLUMNS)}")

print(
    f"Training period: {training_df[TIMESTAMP_COL].iloc[0]} "
    f"to {training_df[TIMESTAMP_COL].iloc[-1]}"
)
print(
    f"Monitoring period: {monitor_df[TIMESTAMP_COL].iloc[0]} "
    f"to {monitor_df[TIMESTAMP_COL].iloc[-1]}"
)

print(
    "Training precedes monitoring:",
    training_df[TIMESTAMP_COL].iloc[-1]
    < monitor_df[TIMESTAMP_COL].iloc[0],
)

/tmp/ipykernel_40035/1527631077.py:54: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  chunk[TIMESTAMP_COL] = pd.to_datetime(
/tmp/ipykernel_40035/1527631077.py:54: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  chunk[TIMESTAMP_COL] = pd.to_datetime(


Training and monitoring data reconstruction: PASS
Eligible observations recovered: 268,801
Training observations: 221,366
Monitoring observations: 47,435
All sensor columns: 51
Selected model features: 42
Training period: 2015-12-28 10:00:00 to 2015-12-31 12:43:42
Monitoring period: 2015-12-31 12:43:43 to 2016-01-01 02:23:17
Training precedes monitoring: True


In [14]:
# Select the established 42 features in the scaler's recorded order.
X_train_raw = training_df[FEATURE_COLUMNS]
X_monitor_raw = monitor_df[FEATURE_COLUMNS]

# Transform using the existing training-fitted scaler.
X_train = scaler.transform(X_train_raw).astype(np.float32)
X_monitor = scaler.transform(X_monitor_raw).astype(np.float32)

# Verify shapes and numerical validity.
assert X_train.shape == (TRAIN_ROWS, len(FEATURE_COLUMNS))
assert X_monitor.shape == (MONITOR_ROWS, len(FEATURE_COLUMNS))
assert np.isfinite(X_train).all()
assert np.isfinite(X_monitor).all()

print("TCN feature scaling: PASS")
print(f"Training matrix: {X_train.shape}, dtype={X_train.dtype}")
print(f"Monitoring matrix: {X_monitor.shape}, dtype={X_monitor.dtype}")
print(f"Training values finite: {np.isfinite(X_train).all()}")
print(f"Monitoring values finite: {np.isfinite(X_monitor).all()}")
print(f"Training range: {X_train.min():.6f} to {X_train.max():.6f}")
print(f"Monitoring range: {X_monitor.min():.6f} to {X_monitor.max():.6f}")

TCN feature scaling: PASS
Training matrix: (221366, 42), dtype=float32
Monitoring matrix: (47435, 42), dtype=float32
Training values finite: True
Monitoring values finite: True
Training range: -0.000074 to 1.000015
Monitoring range: -0.564154 to 1.024334


In [15]:
WINDOW_SIZE = 30

def create_sequences(data, window_size):
    if data.ndim != 2:
        raise ValueError("Expected a 2D array: observations × features.")

    if len(data) < window_size:
        raise ValueError("Not enough observations to create one sequence.")

    sequences = np.lib.stride_tricks.sliding_window_view(
        data,
        window_shape=window_size,
        axis=0,
    )

    # sliding_window_view returns (windows, features, timesteps).
    # Convert explicitly to (windows, timesteps, features).
    sequences = np.moveaxis(sequences, -1, 1)

    return np.ascontiguousarray(sequences, dtype=np.float32)


X_train_seq = create_sequences(X_train, WINDOW_SIZE)
X_monitor_seq = create_sequences(X_monitor, WINDOW_SIZE)

assert X_train_seq.shape == (
    TRAIN_ROWS - WINDOW_SIZE + 1,
    WINDOW_SIZE,
    len(FEATURE_COLUMNS),
)

assert X_monitor_seq.shape == (
    MONITOR_ROWS - WINDOW_SIZE + 1,
    WINDOW_SIZE,
    len(FEATURE_COLUMNS),
)

assert np.isfinite(X_train_seq).all()
assert np.isfinite(X_monitor_seq).all()

# Verify that the first and last windows match their source matrices.
np.testing.assert_array_equal(X_train_seq[0], X_train[:WINDOW_SIZE])
np.testing.assert_array_equal(X_train_seq[-1], X_train[-WINDOW_SIZE:])
np.testing.assert_array_equal(X_monitor_seq[0], X_monitor[:WINDOW_SIZE])
np.testing.assert_array_equal(X_monitor_seq[-1], X_monitor[-WINDOW_SIZE:])

print("TCN sequence creation: PASS")
print(f"Training sequences: {X_train_seq.shape}")
print(f"Monitoring sequences: {X_monitor_seq.shape}")
print(f"Training sequence dtype: {X_train_seq.dtype}")
print(f"Monitoring sequence dtype: {X_monitor_seq.dtype}")
print(f"All training sequences finite: {np.isfinite(X_train_seq).all()}")
print(f"All monitoring sequences finite: {np.isfinite(X_monitor_seq).all()}")

TCN sequence creation: PASS
Training sequences: (221337, 30, 42)
Monitoring sequences: (47406, 30, 42)
Training sequence dtype: float32
Monitoring sequence dtype: float32
All training sequences finite: True
All monitoring sequences finite: True


In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Reproducibility and device selection.
SEED = 42
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# TCN configuration.
N_FEATURES = len(FEATURE_COLUMNS)
WINDOW_SIZE = 30
CHANNELS = [32, 32, 32]
KERNEL_SIZE = 3
DILATIONS = [1, 2, 4]


class CausalConv1d(nn.Module):
    """One-dimensional convolution with left-only padding."""

    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()
        self.left_padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=0,
        )

    def forward(self, x):
        x = F.pad(x, (self.left_padding, 0))
        return self.conv(x)


class TCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()

        self.conv1 = CausalConv1d(
            in_channels, out_channels, kernel_size, dilation
        )
        self.conv2 = CausalConv1d(
            out_channels, out_channels, kernel_size, dilation
        )

        self.activation = nn.ReLU()
        self.residual = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Conv1d(in_channels, out_channels, kernel_size=1)
        )

    def forward(self, x):
        residual = self.residual(x)
        x = self.activation(self.conv1(x))
        x = self.activation(self.conv2(x))
        return self.activation(x + residual)


class TCNAutoencoder(nn.Module):
    def __init__(self, n_features, channels, kernel_size, dilations):
        super().__init__()

        if len(channels) != len(dilations):
            raise ValueError("Each TCN block must have a dilation value.")

        encoder_layers = []
        in_channels = n_features

        for out_channels, dilation in zip(channels, dilations):
            encoder_layers.append(
                TCNBlock(
                    in_channels,
                    out_channels,
                    kernel_size,
                    dilation,
                )
            )
            in_channels = out_channels

        self.encoder = nn.Sequential(*encoder_layers)

        decoder_layers = []
        reversed_channels = list(reversed(channels))
        reversed_dilations = list(reversed(dilations))

        for index, dilation in enumerate(reversed_dilations):
            current_channels = reversed_channels[index]
            next_channels = (
                reversed_channels[index + 1]
                if index + 1 < len(reversed_channels)
                else n_features
            )

            decoder_layers.append(
                CausalConv1d(
                    current_channels,
                    next_channels,
                    kernel_size,
                    dilation,
                )
            )

            if index + 1 < len(reversed_dilations):
                decoder_layers.append(nn.ReLU())

        self.decoder = nn.Sequential(*decoder_layers)

    def forward(self, x):
        # Input: (batch, timesteps, features)
        if x.ndim != 3:
            raise ValueError("Expected input shape (batch, timesteps, features).")

        x = x.transpose(1, 2).contiguous()
        # Convolutional layout: (batch, features, timesteps)

        x = self.encoder(x)
        x = self.decoder(x)

        # Return to (batch, timesteps, features).
        return x.transpose(1, 2).contiguous()


tcn_model = TCNAutoencoder(
    n_features=N_FEATURES,
    channels=CHANNELS,
    kernel_size=KERNEL_SIZE,
    dilations=DILATIONS,
).to(DEVICE)

# Test a small batch before any training.
test_input = torch.from_numpy(X_train_seq[:8]).to(DEVICE)

with torch.no_grad():
    test_output = tcn_model(test_input)

assert test_output.shape == test_input.shape, (
    f"Shape mismatch: input {tuple(test_input.shape)}, "
    f"output {tuple(test_output.shape)}"
)
assert torch.isfinite(test_output).all(), "Model output contains non-finite values."

print("TCN architecture test: PASS")
print(f"Device: {DEVICE}")
print(f"Input shape: {tuple(test_input.shape)}")
print(f"Output shape: {tuple(test_output.shape)}")
print(f"Trainable parameters: {sum(p.numel() for p in tcn_model.parameters() if p.requires_grad):,}")
print(f"All output values finite: {torch.isfinite(test_output).all().item()}")

TCN architecture test: PASS
Device: cpu
Input shape: (8, 30, 42)
Output shape: (8, 30, 42)
Trainable parameters: 31,242
All output values finite: True


In [17]:
# Verify the TCN architecture configuration and receptive field.

receptive_field = 1

for dilation in DILATIONS:
    # Each block contains two convolutions at this dilation.
    receptive_field += 2 * (KERNEL_SIZE - 1) * dilation

print("TCN architecture verification")
print(f"Input window: {WINDOW_SIZE} timesteps")
print(f"Kernel size: {KERNEL_SIZE}")
print(f"Channels: {CHANNELS}")
print(f"Block dilations: {DILATIONS}")
print("Convolutions per block: 2")
print(f"Encoder receptive field: {receptive_field} timesteps")
print(f"Receptive field covers full input window: {receptive_field >= WINDOW_SIZE}")

assert receptive_field == 29
assert receptive_field < WINDOW_SIZE

TCN architecture verification
Input window: 30 timesteps
Kernel size: 3
Channels: [32, 32, 32]
Block dilations: [1, 2, 4]
Convolutions per block: 2
Encoder receptive field: 29 timesteps
Receptive field covers full input window: False
